# Experiment XX: [Experiment Name]

**Date:** YYYY-MM-DD  
**Author:** Your Name  
**Objective:** Brief objective statement

---

## 📋 Experiment Overview

### Hypothesis
State your hypothesis here.

### Key Changes from Previous Experiment
1. Change 1
2. Change 2
3. Change 3

### Configuration
- **Model:** Model architecture
- **Preprocessing:** Preprocessing steps
- **Dataset:** REFUGE (cropped masks)
- **Training samples:** 320
- **Validation samples:** 80
- **Test samples:** 400

### Hyperparameters
```python
{
    'epochs': 50,
    'batch_size': 8,
    'learning_rate': 1e-4,
    # Add more...
}
```

---

## 1️⃣ Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import json
from datetime import datetime

# Project imports
# from models.your_model import YourModel
from data_loader.dataset import RetinaDataset, RetinaDatasetTest
from utils.metrics import batch_metrics, calculate_cdr

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2️⃣ Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Model
    'model': 'your_model',
    'base_features': 64,
    
    # Data
    'data_dir': '../../datasets/REFUGE',
    'train_csv': '../../datasets/REFUGE/REFUGETrain.csv',
    'val_csv': '../../datasets/REFUGE/REFUGE1Val.csv',
    'test_csv': '../../datasets/REFUGE/REFUGE1Test.csv',
    'use_cropped': True,
    'use_clahe': False,  # Change as needed
    
    # Training
    'epochs': 50,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    
    # Output
    'output_dir': './results',
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/visualizations", exist_ok=True)

# Save configuration
with open(f"{CONFIG['output_dir']}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=4)

print("✅ Configuration saved")
print(json.dumps(CONFIG, indent=2))

## 3️⃣ Data Loading

In [ ]:
# Create datasets
print("Loading datasets...")

train_dataset = RetinaDataset(
    csv_file=CONFIG['train_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

val_dataset = RetinaDataset(
    csv_file=CONFIG['val_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

test_dataset = RetinaDatasetTest(
    csv_file=CONFIG['test_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=CONFIG['num_workers'], pin_memory=True)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")
print(f"✅ Test samples: {len(test_dataset)}")

## 4️⃣ Model Definition

In [ ]:
# Create model
# model = YourModel(...).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'],
                       weight_decay=CONFIG['weight_decay'])

print("✅ Loss function and optimizer defined")

## 5️⃣ Training

In [ ]:
# Training functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    epoch_dice = 0
    
    pbar = tqdm(loader, desc='Training')
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return epoch_loss / len(loader), epoch_dice / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    epoch_dice_disc = 0
    epoch_dice_cup = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
            epoch_dice_disc += metrics['dice_disc']
            epoch_dice_cup += metrics['dice_cup']
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return {
        'loss': epoch_loss / len(loader),
        'dice_mean': epoch_dice / len(loader),
        'dice_disc': epoch_dice_disc / len(loader),
        'dice_cup': epoch_dice_cup / len(loader),
    }

print("✅ Training functions defined")

In [ ]:
# Training loop
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [],
           'val_dice_disc': [], 'val_dice_cup': []}
best_dice = 0.0
start_time = datetime.now()

print(f"\n{'='*60}")
print("Starting training...")
print(f"{'='*60}\n")

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 60)
    
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_metrics['loss'])
    history['val_dice'].append(val_metrics['dice_mean'])
    history['val_dice_disc'].append(val_metrics['dice_disc'])
    history['val_dice_cup'].append(val_metrics['dice_cup'])
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"  Val Loss:   {val_metrics['loss']:.4f} | Val Dice:   {val_metrics['dice_mean']:.4f}")
    print(f"  Val Disc:   {val_metrics['dice_disc']:.4f} | Val Cup:    {val_metrics['dice_cup']:.4f}")
    
    if val_metrics['dice_mean'] > best_dice:
        best_dice = val_metrics['dice_mean']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'args': CONFIG,
        }, f"{CONFIG['output_dir']}/best_model.pth")
        print(f"  ✅ Best model saved (Dice: {best_dice:.4f})")

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds() / 3600

print(f"\n{'='*60}")
print(f"Training completed in {training_time:.2f} hours")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"{'='*60}")

### 📈 Training Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_dice'], label='Train Dice', linewidth=2)
axes[1].plot(history['val_dice'], label='Val Dice (Overall)', linewidth=2)
axes[1].plot(history['val_dice_disc'], label='Val Dice (Disc)', linewidth=2, linestyle='--')
axes[1].plot(history['val_dice_cup'], label='Val Dice (Cup)', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Training and Validation Dice Scores')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

with open(f"{CONFIG['output_dir']}/training_history.json", 'w') as f:
    json.dump(history, f, indent=4)

print("✅ Training curves saved")

## 6️⃣ Testing

In [ ]:
# Load best model
print("Loading best model...")
checkpoint = torch.load(f"{CONFIG['output_dir']}/best_model.pth", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best model loaded (from epoch {checkpoint['epoch']+1})")
print(f"   Validation Dice: {checkpoint['metrics']['dice_mean']:.4f}")

In [ ]:
# Test model
print("\nTesting model on test set...")

test_metrics = {'dice_mean': [], 'dice_disc': [], 'dice_cup': [], 'cdr_mae': []}
all_predictions = []
all_masks = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing')
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        metrics = batch_metrics(outputs, masks)
        
        test_metrics['dice_mean'].append(metrics['dice_mean'])
        test_metrics['dice_disc'].append(metrics['dice_disc'])
        test_metrics['dice_cup'].append(metrics['dice_cup'])
        test_metrics['cdr_mae'].append(metrics['cdr_mae'])
        
        preds = torch.sigmoid(outputs).cpu()
        all_predictions.append(preds)
        all_masks.append(masks.cpu())
        
        pbar.set_postfix({'dice': f"{metrics['dice_mean']:.4f}", 'cdr_mae': f"{metrics['cdr_mae']:.4f}"})

test_results = {
    'dice_mean': np.mean(test_metrics['dice_mean']),
    'dice_disc': np.mean(test_metrics['dice_disc']),
    'dice_cup': np.mean(test_metrics['dice_cup']),
    'cdr_mae': np.mean(test_metrics['cdr_mae']),
}

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Overall Dice Score:     {test_results['dice_mean']:.4f} ({test_results['dice_mean']*100:.2f}%)")
print(f"Disc Dice Score:        {test_results['dice_disc']:.4f} ({test_results['dice_disc']*100:.2f}%)")
print(f"Cup Dice Score:         {test_results['dice_cup']:.4f} ({test_results['dice_cup']*100:.2f}%)")
print(f"CDR Mean Absolute Error: {test_results['cdr_mae']:.4f}")
print("="*60)

with open(f"{CONFIG['output_dir']}/test_results.json", 'w') as f:
    json.dump(test_results, f, indent=4)

print("\n✅ Test results saved")

## 7️⃣ Visualizations

In [ ]:
# Concatenate predictions
all_predictions = torch.cat(all_predictions, dim=0)
all_masks = torch.cat(all_masks, dim=0)

# Select 4 samples
np.random.seed(42)
vis_indices = np.random.choice(len(test_dataset), 4, replace=False)

# Create visualizations
fig, axes = plt.subplots(4, 5, figsize=(20, 16))

for i, idx in enumerate(vis_indices):
    image, mask_gt = test_dataset[idx]
    pred = all_predictions[idx]
    
    # Denormalize image
    img_np = image.numpy().transpose(1, 2, 0)
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    disc_gt = mask_gt[0].numpy()
    cup_gt = mask_gt[1].numpy()
    disc_pred = (pred[0].numpy() > 0.5).astype(float)
    cup_pred = (pred[1].numpy() > 0.5).astype(float)
    
    disc_dice = 2 * (disc_pred * disc_gt).sum() / (disc_pred.sum() + disc_gt.sum() + 1e-8)
    cup_dice = 2 * (cup_pred * cup_gt).sum() / (cup_pred.sum() + cup_gt.sum() + 1e-8)
    overall_dice = (disc_dice + cup_dice) / 2
    
    cdr_gt = cup_gt.sum() / (disc_gt.sum() + 1e-8)
    cdr_pred = cup_pred.sum() / (disc_pred.sum() + 1e-8)
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Sample {idx}\nOverall Dice: {overall_dice:.3f}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(disc_gt, cmap='gray')
    axes[i, 1].set_title(f'GT Disc\n(CDR: {cdr_gt:.3f})')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(cup_gt, cmap='gray')
    axes[i, 2].set_title('GT Cup')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(disc_pred, cmap='gray')
    axes[i, 3].set_title(f'Pred Disc\nDice: {disc_dice:.3f}')
    axes[i, 3].axis('off')
    
    axes[i, 4].imshow(cup_pred, cmap='gray')
    axes[i, 4].set_title(f'Pred Cup\nDice: {cup_dice:.3f}\nCDR: {cdr_pred:.3f}')
    axes[i, 4].axis('off')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/visualizations/predictions.png", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualizations saved")

## 8️⃣ Results Summary

### Key Findings

**Performance Metrics:**
- Overall Dice Score: See test results above
- Optic Disc Dice: See test results above
- Optic Cup Dice: See test results above
- CDR MAE: See test results above

**Training Details:**
- Training time: See output above
- Best validation epoch: See checkpoint
- Model parameters: See model definition

### Comparison with Previous Experiments

| Metric | Previous | This Experiment | Change |
|--------|----------|----------------|--------|
| Overall Dice | X.XX% | X.XX% | +/-X.XX% |
| Disc Dice | X.XX% | X.XX% | +/-X.XX% |
| Cup Dice | X.XX% | X.XX% | +/-X.XX% |
| CDR MAE | X.XXX | X.XXX | +/-X.XXX |

### Observations

1. Observation 1
2. Observation 2
3. Observation 3

### Next Steps

Based on these results, next steps could include:
- Idea 1
- Idea 2
- Idea 3

---

**Experiment Status:** ✅/❌/🚧 [Complete/Failed/In Progress]